# Three-way kSZ D_ell comparison

Stitched lightcone vs. coeval (Cain-direct) vs. coeval (Georgiev Eq.10 reconstruction), all from the same fiducial 800 Mpc/128^3 run. Run scripts 02 and 03 first (--config configs/fiducial.yaml), then rsync `data/products/` back to desktop before running this notebook -- see session notes for the exact rsync command.

The native lightcone (script 01) is loaded too, if present, purely for context -- it's the known outlier (~1000x too large relative to the other two methods, isolated to something in the native run_lightcone() path, not the shared map/FFT machinery) and is plotted in a deliberately muted style so it doesn't visually dominate the real comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from ksz_pipeline.plotting.styles import PNG_STYLE, PDF_STYLE, save_pdf_png

## Load products

In [ ]:
products = {}

for label, fname in [
    ('stitched',        'data/products/ksz_Dl_stitched.npz'),
    ('coeval_direct',   'data/products/ksz_Dl_coeval.npz'),
    ('coeval_georgiev', 'data/products/ksz_Dl_coeval_georgiev.npz'),
    ('native_lightcone','data/products/ksz_Dl_lightcone.npz'),  # context only, see markdown above
]:
    try:
        products[label] = np.load(fname)
        print(f'Loaded {label} <- {fname}')
    except FileNotFoundError:
        print(f'NOT FOUND (skipping): {fname}')


## D_ell comparison

Watch for: do stitched and coeval_direct land in the same rough family? (They should -- that agreement is what isolated the native lightcone as the outlier in the first place.) Does coeval_georgiev sit systematically above coeval_direct, consistent with the Wick/Gaussian reconstruction overshoot seen at quicktest scale -- or has that pattern changed at full resolution?

In [ ]:
def plot_func(ax):
    style_map = {
        'stitched':         dict(color='darkgreen', marker='o', ls='-',  label=r'This work: Stitched lightcone $D_\ell$'),
        'coeval_direct':     dict(color='crimson',    marker='o', ls='--', label=r'This work: Coeval (direct) $D_\ell$'),
        'coeval_georgiev':   dict(color='darkorange',  marker='^', ls='-.', label=r'This work: Coeval (Georgiev Eq.10) $D_\ell$'),
    }
    for label, style in style_map.items():
        if label not in products:
            continue
        d = products[label]
        yerr = d['sigma_Dl'] if 'sigma_Dl' in d.files else d['Dl_err'] if 'Dl_err' in d.files else None
        ax.errorbar(d['ell'], d['Dl'], yerr=yerr, lw=1.8, ms=5, capsize=3, **style)

    if 'native_lightcone' in products:
        d = products['native_lightcone']
        ax.plot(d['ell'], d['Dl'], color='gray', ls=':', lw=1.0, alpha=0.5,
                label=r'Native lightcone $D_\ell$ (known outlier -- context only)')

    ax.errorbar(3000, 1.1, yerr=[[0.7],[1.0]], fmt='s', ms=8, capsize=5,
                color='red', label='Reichardt+2021')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'Multipole $\ell$'); ax.set_ylabel(r'$D_\ell\ [\mu{\rm K}^2]$')
    ax.set_xlim(1e2, 1e4)
    ax.legend(loc='upper left', fontsize=11)

with mpl.rc_context(PNG_STYLE):
    fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
    plot_func(ax)
    plt.show()


## Save (PDF + PNG, matches this project's other figures)

In [ ]:
import os
os.makedirs('data/plots', exist_ok=True)
save_pdf_png(plot_func, 'data/plots', 'three_way_Dell_comparison',
             title='Stitched vs Coeval (Cain/Georgiev) D_ell')
print('Saved -> data/plots/three_way_Dell_comparison.pdf/.png')